In [1]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        """
        初始化
        dim: 词向量的维度 (比如 4096)
        eps: 一个极小的数，防止除以 0 报错
        """
        super().__init__()
        self.eps = eps
        self.dim = dim
        
        # 1. 定义一个可学习的缩放参数 gamma (初始化为全 1)
        # 为什么需要这个？因为单纯把数据压扁可能会丢失特征，让模型自己学着把数据拉伸回来。
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        """
        前向传播
        x 形状: (batch, seq_len, dim)
        """
        # ==========================================
        # 第 1 步：算均方根 (RMS)
        # ==========================================
        # 1.1 先求平方：x^2
        x_squared = x.pow(2)
        
        # 1.2 求均值 (沿着最后一个维度 dim 求均值)
        # keepdim=True 极其重要！它的作用是保持维度，形状从 (B, S, D) 变成 (B, S, 1)
        # 这样后面才能用广播机制，把整个 D 维度的数都除以这个均值
        mean_squared = x_squared.mean(dim=-1, keepdim=True)
        
        # 1.3 加上 eps 并开根号取倒数 (rsqrt 就是 1 / 根号x)
        # 对应的公式： 1 / sqrt( mean(x^2) + eps )
        rms = torch.rsqrt(mean_squared + self.eps)
        
        # ==========================================
        # 第 2 步：归一化 并 缩放
        # ==========================================
        # 2.1 原始输入乘以 rms (把数据压扁)
        x_normalized = x * rms
        
        # 2.2 乘以可学习的权重参数 (让模型自己决定要放大还是缩小)
        out = self.weight * x_normalized
        
        return out


In [2]:
# 假设 batch=1, seq_len=2, dim=4
x = torch.tensor([[[1.0, 2.0, 3.0, 4.0],   # 第 1 个 token
                   [2.0, 4.0, 6.0, 8.0]]]) # 第 2 个 token

rmsnorm = RMSNorm(dim=4)
out = rmsnorm(x)

print("输入:\n", x)
print("输出:\n", out)

# 我们手动算第一行验证一下：
# 1. 平方: [1, 4, 9, 16]
# 2. 均值: (1+4+9+16) / 4 = 30 / 4 = 7.5
# 3. 均方根: 1 / sqrt(7.5 + 1e-6) ≈ 0.3651
# 4. 乘以原值: [1*0.3651, 2*0.3651, 3*0.3651, 4*0.3651] 
# 结果约为: [0.3651, 0.7303, 1.0954, 1.4606]
# （代码输出的结果应该和这个极度接近）

输入:
 tensor([[[1., 2., 3., 4.],
         [2., 4., 6., 8.]]])
输出:
 tensor([[[0.3651, 0.7303, 1.0954, 1.4606],
         [0.3651, 0.7303, 1.0954, 1.4606]]], grad_fn=<MulBackward0>)


    def forward(self, x):
        # 把输入临时转成 fp32 算，算完再转回输入原本的类型 (如 fp16)
        return (self.weight * self._norm(x.float())).type_as(x)